# Task Scheduler (topological sort with cycle detection)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Graphs, Hash Tables · **Difficulty/Frequency:** Common (5/10)

## Concepts

**What this problem is really testing:**
- Recognising "must run before" as a **directed graph**, and the answer as a **topological sort**
- **Kahn's algorithm** — and the fact that its cycle detection is *free*, not an extra pass
- Building the **reverse** adjacency map, which is what turns O(V²) into O(V + E)

**First-principles primer — what is each piece?**

- **Directed graph.** Vertices are tasks; an edge `A -> B` means "A must finish before B". Direction is the whole content of a dependency.
- **DAG (directed acyclic graph).** A directed graph with no cycles. Only a DAG can be scheduled: if A needs B and B needs A, neither can ever start. So "detect circular dependencies" is really "verify this graph is a DAG".
- **Topological sort.** An ordering of the vertices in which every edge points forward — every task appears after all of its dependencies. There is usually more than one valid answer, which is why tie-breaking matters.
- **In-degree.** How many edges point *into* a vertex — here, how many dependencies a task is still waiting on. **In-degree zero means runnable right now.**
- **Kahn's algorithm.** Repeatedly: take a task with in-degree 0, run it, and decrement the in-degree of everything that depended on it. Anything that drops to 0 becomes runnable.

**The two insights that make it O(V + E):**

1. **Keep a running in-degree count instead of rechecking.** The naive loop rescans every unexecuted task after every completion — O(V²) at best. A counter per task means you only touch what actually changed.
2. **Store the *reverse* edges (`dependents`), not just the forward ones.** When task A finishes, you need "who was waiting on A?" — that is the reverse direction. Storing only `deps` (forward) would force a scan of every task to find them. Building both directions at insert time is what makes the decrement step O(1) per edge.

**Cycle detection comes free.** A task inside a cycle can *never* reach in-degree 0 — something in the cycle is always still ahead of it. So the queue empties with work left over, and `len(executed) < len(tasks)` is the complete test. **No second pass, no visited-set colouring.** That is the elegance of Kahn's, and it is the thing to say out loud.

**About "in the order they were submitted".** Taken literally this is impossible — if task 1 depends on task 2, submission order violates the dependency. The honest reading is: *respect dependencies first; among tasks that are simultaneously runnable, prefer the earliest-submitted*.

Getting that tie-break right has a concrete consequence: **`dependents` must be an insertion-ordered container, not a `set`.** Python dicts preserve insertion order, but sets do **not** — their iteration order is hash-based. Store the reverse edges in a `set` and the relaxation loop enqueues newly-runnable tasks in an arbitrary order, so the output is still a *valid* topological order but not a *deterministic* one. Two runs can differ, and tests become flaky for reasons that look like magic. A `list` fixes it, and duplicates are impossible because each edge is added exactly once.

**Simple worked example.** `A` (no deps), `B` needs `A`, `C` needs `A`, `D` needs `B` and `C` — a diamond:

| step | queue | run | in-degrees after |
|---|---|---|---|
| start | `[A]` | — | `A:0 B:1 C:1 D:2` |
| 1 | `[B, C]` | `A` | `B:0 C:0 D:2` |
| 2 | `[C]` | `B` | `C:0 D:1` |
| 3 | `[D]` | `C` | `D:0` |
| 4 | `[]` | `D` | — |

Order: `A, B, C, D`. Note `D` waited for **both** parents — the in-degree of 2 is exactly what encodes that.

## Problem Statement

| Method | Behaviour |
|---|---|
| `add_task(task_id, dependencies)` | Register a task; its dependencies must run before it |
| `execute_all()` | Run everything in a valid order, respecting dependencies; **detect circular dependencies** and fail with useful feedback |

Among tasks that are simultaneously runnable, prefer the earliest submitted.

**Example**

```python
s = TaskScheduler()
s.add_task("A")
s.add_task("B", ["A"])
s.add_task("C", ["A"])
s.add_task("D", ["B", "C"])
s.execute_all()          # -> ["A", "B", "C", "D"]
```

### Approach 1 — Naive (rescan for a runnable task, repeatedly)

**Idea:** loop until everything has run. Each pass, walk every unexecuted task and check whether *all* of its dependencies are already done; run the first such task and start over.

It is the direct statement of the requirement, and it is correct — including the cycle case, where a full pass finds nothing runnable and you can stop. But it rechecks the entire graph after every single completion.

**Time complexity:** **O(V² + V·E)** — V rounds, each scanning up to V tasks and re-checking their dependency sets.

**Space complexity:** O(V + E).

In [ ]:
from typing import Dict, List, Optional, Set


class NaiveTaskScheduler:
    """Baseline: correct, but rescans every remaining task after each execution."""

    def __init__(self) -> None:
        self.order: List[str] = []                    # submission order
        self.deps: Dict[str, Set[str]] = {}

    def add_task(self, task_id: str, dependencies: Optional[List[str]] = None) -> None:
        if task_id in self.deps:
            raise ValueError(f"Task '{task_id}' already exists.")
        for d in dependencies or []:
            if d not in self.deps:
                raise ValueError(f"Dependency '{d}' not found for task '{task_id}'.")
        self.order.append(task_id)
        self.deps[task_id] = set(dependencies or [])

    def execute_all(self) -> List[str]:
        done: Set[str] = set()
        executed: List[str] = []
        while len(executed) < len(self.order):
            progressed = False
            for t in self.order:                      # O(V) scan, repeated V times
                if t not in done and self.deps[t] <= done:   # all dependencies satisfied?
                    executed.append(t)
                    done.add(t)
                    progressed = True
                    break                             # restart the scan from the top
            if not progressed:                        # a full pass found nothing runnable
                remaining = [t for t in self.order if t not in done]
                raise RuntimeError(f"Circular dependency detected among tasks: {remaining}")
        return executed

### Approach 2 — Optimal (Kahn's algorithm)

**Idea:** stop rediscovering what is runnable. Track it.

- `in_degree[t]` — how many dependencies `t` is still waiting on.
- `dependents[t]` — the **reverse** edges: who is waiting on `t`.

Seed a queue with every task at in-degree 0, in submission order. Then repeat: pop, execute, and decrement each dependent's counter; anything hitting 0 joins the queue.

**Two design decisions worth defending aloud:**

- **A `deque`, not a list.** `list.pop(0)` is O(n) because it shifts every remaining element; `deque.popleft()` is O(1). Using a list here quietly reintroduces an O(V²) term into an otherwise O(V + E) algorithm.
- **`dependents` is a `list`, not a `set`.** Sets iterate in hash order, so relaxing a finished task's edges would enqueue its newly-runnable dependents in an arbitrary order — a valid topological sort, but a non-deterministic one. The list keeps submission order, which is exactly the tie-break the problem asks for.
- **Unknown dependencies are rejected at `add_task` time.** The alternative — recording an edge to a task that may never arrive — pushes the failure to `execute_all`, where it is indistinguishable from a cycle. Failing at the point of the mistake gives a far better error message. State the contract either way.

**Cycle detection:** if the queue drains with `len(executed) < len(tasks)`, whatever is left is exactly the set of tasks trapped in (or downstream of) a cycle. Reporting *which* tasks they are is what makes the error actionable.

**Time complexity:** **O(V + E)** — every task is enqueued and dequeued once; every edge is relaxed once.

**Space complexity:** O(V + E) for the two adjacency maps, plus O(V) for the counters and queue.

In [ ]:
from collections import deque


class TaskScheduler:
    """Kahn's algorithm: run whatever has no unmet dependencies, then relax its edges."""

    def __init__(self) -> None:
        self.tasks: Dict[str, object] = {}             # id -> payload (callable or the id itself)
        self.deps: Dict[str, Set[str]] = {}            # id -> what it waits on   (forward edges)
        self.dependents: Dict[str, List[str]] = {}     # id -> who waits on it    (REVERSE edges)

    def add_task(self, task_id: str, dependencies: Optional[List[str]] = None,
                 payload: object = None) -> None:
        if task_id in self.tasks:
            raise ValueError(f"Task '{task_id}' already exists.")
        dependencies = list(dependencies or [])
        for d in dependencies:
            if d not in self.tasks:                    # reject forward references at the SOURCE
                raise ValueError(f"Dependency '{d}' not found for task '{task_id}'.")
        if task_id in dependencies:
            raise ValueError(f"Task '{task_id}' cannot depend on itself.")

        self.tasks[task_id] = task_id if payload is None else payload
        self.deps[task_id] = set(dependencies)
        self.dependents[task_id] = []
        for d in dependencies:
            # A LIST, not a set: iteration order here decides which task is enqueued first
            # when several become runnable at once, and a set's order is hash-based.
            self.dependents[d].append(task_id)         # build BOTH directions while we are here

    def execute_all(self) -> List[str]:
        in_degree = {t: len(self.deps[t]) for t in self.tasks}
        # dict iteration is insertion order => ties break by submission order, by construction
        queue = deque(t for t in self.tasks if in_degree[t] == 0)
        executed: List[str] = []

        while queue:
            t = queue.popleft()                        # deque: O(1). A list would be O(V) here.
            executed.append(t)
            payload = self.tasks[t]
            if callable(payload):
                payload()
            for dep in self.dependents[t]:             # only the tasks actually affected
                in_degree[dep] -= 1
                if in_degree[dep] == 0:                # just became runnable
                    queue.append(dep)

        if len(executed) < len(self.tasks):            # the queue drained with work left over
            done = set(executed)
            stuck = [t for t in self.tasks if t not in done]
            raise RuntimeError(f"Circular dependency detected among tasks: {stuck}")
        return executed

### Follow-up 1 — strict submission-order tie-breaking (a heap, not a queue)

**Idea:** Kahn's FIFO queue gives a *valid* order, but not necessarily the **earliest-submitted-first** one. The two policies genuinely differ:

- **FIFO (`deque`)** — "first to *become* runnable, first to run". A task that became runnable in round 1 always precedes one that became runnable in round 2, even if the latter was submitted first.
- **Earliest-submitted (`heap`)** — among *everything currently runnable*, always pick the lowest submission index, regardless of when it became ready.

Concretely, with `t0`, `t1` (depends on `t0`), and `t2` (no dependencies):

| policy | order | why |
|---|---|---|
| FIFO | `t0, t2, t1` | `t0` and `t2` are both ready at the start, so both are queued before `t1` becomes ready |
| earliest-submitted | `t0, t1, t2` | once `t0` runs, `t1` is runnable and has a lower index than `t2` |

Both satisfy every dependency. If the spec really means "prefer the earliest submitted", you need the **heap** — swapping the `deque` for a `heapq` keyed by submission index is a two-line change, at the cost of O(V log V) instead of O(V).

**Time complexity:** O((V + E) log V).

**Space complexity:** O(V + E).

In [ ]:
import heapq


class SubmissionOrderScheduler(TaskScheduler):
    """Among ALL currently-runnable tasks, always run the earliest-submitted one."""

    def execute_all(self) -> List[str]:
        index = {t: i for i, t in enumerate(self.tasks)}      # submission index
        in_degree = {t: len(self.deps[t]) for t in self.tasks}
        heap = [(index[t], t) for t in self.tasks if in_degree[t] == 0]
        heapq.heapify(heap)                                  # a HEAP, not a FIFO queue
        executed: List[str] = []

        while heap:
            _, t = heapq.heappop(heap)                       # globally lowest submission index
            executed.append(t)
            payload = self.tasks[t]
            if callable(payload):
                payload()
            for dep in self.dependents[t]:
                in_degree[dep] -= 1
                if in_degree[dep] == 0:
                    heapq.heappush(heap, (index[dep], dep))

        if len(executed) < len(self.tasks):
            done = set(executed)
            stuck = [t for t in self.tasks if t not in done]
            raise RuntimeError(f"Circular dependency detected among tasks: {stuck}")
        return executed

### Follow-up 2 — parallel execution (which tasks can run *at the same time*?)

**Idea:** Kahn's already computes this, and most people miss it. Every task in the queue at the start of a round has **zero unmet dependencies**, so they are mutually independent and can all run concurrently.

Draining the queue **one whole level at a time** partitions the tasks into *waves*. The number of waves is the **critical path length** — the longest chain of dependencies — and it is the hard floor on how fast the whole job can finish, no matter how many workers you have. The widest wave tells you the most parallelism you could ever use.

For the diamond above: `[[A], [B, C], [D]]` — three waves. Even with a thousand machines it takes three rounds, because `D` genuinely cannot start until `B` and `C` are done.

**Time complexity:** O(V + E), unchanged.

**Space complexity:** O(V).

In [ ]:
class ParallelTaskScheduler(TaskScheduler):
    """Group tasks into waves: everything in one wave can run concurrently."""

    def execution_waves(self) -> List[List[str]]:
        in_degree = {t: len(self.deps[t]) for t in self.tasks}
        ready = [t for t in self.tasks if in_degree[t] == 0]
        waves: List[List[str]] = []
        seen = 0

        while ready:
            waves.append(ready)                        # this whole wave is mutually independent
            seen += len(ready)
            nxt: List[str] = []
            for t in ready:                            # drain the ENTIRE level before moving on
                for dep in self.dependents[t]:
                    in_degree[dep] -= 1
                    if in_degree[dep] == 0:
                        nxt.append(dep)
            ready = nxt

        if seen < len(self.tasks):
            done = {t for w in waves for t in w}
            raise RuntimeError(
                f"Circular dependency detected among tasks: "
                f"{[t for t in self.tasks if t not in done]}"
            )
        return waves

### Follow-up 3 — detecting a cycle at `add_task` time

**Idea:** the version above only discovers a cycle at `execute_all`, potentially long after the offending call. Rejecting the edge that *creates* the cycle points straight at the mistake.

Here it is nearly free. Because `add_task` requires every dependency to **already exist**, a new task can only ever point at *older* tasks — and edges that always run older-to-newer cannot close a loop. So under that contract the graph **cannot** contain a cycle at all.

That makes the interesting version the one where the contract is relaxed: `add_dependency(a, b)` between two existing tasks *can* create a cycle, and the check is "would `b` already reach `a`?" — one DFS, O(V + E) per added edge. Worth it when adds are rare and you want failures reported at their source; not worth it when you add millions of edges and validate once at the end.

**Time complexity:** O(V + E) per dependency added.

**Space complexity:** O(V) for the DFS stack and visited set.

In [ ]:
class EagerCycleScheduler(TaskScheduler):
    """Allows edges between existing tasks, and rejects any edge that would close a cycle."""

    def add_dependency(self, task_id: str, depends_on: str) -> None:
        if task_id not in self.tasks or depends_on not in self.tasks:
            raise ValueError("both tasks must already exist")
        if task_id == depends_on:
            raise ValueError(f"Task '{task_id}' cannot depend on itself.")
        # Adding "depends_on -> task_id" is safe unless task_id can ALREADY reach depends_on.
        if self._reaches(task_id, depends_on):
            raise ValueError(
                f"Adding '{task_id}' -> depends on '{depends_on}' would create a cycle."
            )
        self.deps[task_id].add(depends_on)
        self.dependents[depends_on].append(task_id)

    def _reaches(self, src: str, dst: str) -> bool:
        """Is dst reachable from src by following 'dependents' edges? Iterative DFS."""
        stack, seen = [src], {src}
        while stack:
            node = stack.pop()
            if node == dst:
                return True
            for nxt in self.dependents[node]:
                if nxt not in seen:
                    seen.add(nxt)
                    stack.append(nxt)
        return False

## Verification

Check the diamond, the tie-breaking rule, and — most importantly — that **every** returned order is genuinely valid (every dependency appears before its dependent), not just that it matches one expected list.

In [ ]:
import random


def build(cls, spec):
    s = cls()
    for tid, deps in spec:
        s.add_task(tid, deps)
    return s


def assert_valid_order(order, spec):
    """The real contract: every dependency must appear before the task that needs it."""
    position = {t: i for i, t in enumerate(order)}
    assert len(order) == len(spec) == len(set(order)), "each task exactly once"
    for tid, deps in spec:
        for d in deps or []:
            assert position[d] < position[tid], f"{d} must run before {tid}: {order}"


DIAMOND = [("A", []), ("B", ["A"]), ("C", ["A"]), ("D", ["B", "C"])]

# --- The diamond, on both implementations ---
for cls in (TaskScheduler, NaiveTaskScheduler):
    order = build(cls, DIAMOND).execute_all()
    assert order == ["A", "B", "C", "D"], (cls.__name__, order)
    assert_valid_order(order, DIAMOND)

# --- A dependency submitted before its dependent, but declared later ---
chain = [("setup", []), ("build", ["setup"]), ("test", ["build"]), ("deploy", ["test"])]
for cls in (TaskScheduler, NaiveTaskScheduler):
    assert build(cls, chain).execute_all() == ["setup", "build", "test", "deploy"], cls.__name__

# --- Independent tasks come out in submission order ---
indep = [("z", []), ("y", []), ("x", [])]
for cls in (TaskScheduler, NaiveTaskScheduler):
    assert build(cls, indep).execute_all() == ["z", "y", "x"], cls.__name__

# --- Tie-breaking: when several become runnable at once, earliest submitted wins ---
tie = [("root", []), ("first", ["root"]), ("second", ["root"]), ("third", ["root"])]
assert build(TaskScheduler, tie).execute_all() == ["root", "first", "second", "third"]

# --- FIFO and earliest-submitted are DIFFERENT policies, and both are valid ---
policies = [("t0", []), ("t1", ["t0"]), ("t2", [])]
assert build(TaskScheduler, policies).execute_all() == ["t0", "t2", "t1"], "FIFO by readiness"
assert build(SubmissionOrderScheduler, policies).execute_all() == ["t0", "t1", "t2"],     "heap: earliest submitted among all runnable"
for cls in (TaskScheduler, SubmissionOrderScheduler, NaiveTaskScheduler):
    assert_valid_order(build(cls, policies).execute_all(), policies)

# --- Edge cases ---
assert TaskScheduler().execute_all() == [], "an empty scheduler runs nothing"
single = TaskScheduler(); single.add_task("only")
assert single.execute_all() == ["only"]

# --- Cycle detection ---
def force_dependency(sched, task_id, depends_on):
    """Wire an edge directly, bypassing add_task's validation.

    add_task can only point at tasks that already exist, so under its own contract a
    cycle is impossible to build. To exercise execute_all's detection we have to create
    an invalid graph deliberately.
    """
    sched.deps[task_id].add(depends_on)
    sched.dependents[depends_on].append(task_id)


s = TaskScheduler()
s.add_task("a"); s.add_task("b", ["a"]); s.add_task("c", ["b"])
force_dependency(s, "a", "c")                    # a now depends on c: a -> b -> c -> a
try:
    s.execute_all()
except RuntimeError as e:
    assert "Circular" in str(e)
    for t in ("a", "b", "c"):
        assert t in str(e), f"the error must name the stuck tasks: {e}"
else:
    raise AssertionError("a cycle must be detected")

# A cycle must not hide the tasks that CAN run - they are simply reported as not-stuck
s2 = TaskScheduler()
for t in ("ok", "x", "y"):
    s2.add_task(t)
force_dependency(s2, "x", "y"); force_dependency(s2, "y", "x")
try:
    s2.execute_all()
except RuntimeError as e:
    stuck_list = str(e).split(":", 1)[1]
    assert "ok" not in stuck_list, f"'ok' is runnable and must not be listed as stuck: {e}"
    assert "x" in stuck_list and "y" in stuck_list, e
else:
    raise AssertionError("a cycle must be detected")

# --- Contract violations are rejected at the point of the mistake ---
s = TaskScheduler(); s.add_task("a")
for bad, msg in [
    (lambda: s.add_task("a"), "duplicate task id"),
    (lambda: s.add_task("b", ["missing"]), "unknown dependency"),
    (lambda: s.add_task("c", ["c"]), "self-dependency"),
]:
    try:
        bad()
    except ValueError:
        pass
    else:
        raise AssertionError(f"{msg} should raise ValueError")

# --- Tasks as callables: they actually run, in order ---
log = []
s = TaskScheduler()
s.add_task("first", None, payload=lambda: log.append("first"))
s.add_task("second", ["first"], payload=lambda: log.append("second"))
s.execute_all()
assert log == ["first", "second"]

# --- Parallel waves ---
p = build(ParallelTaskScheduler, DIAMOND)
assert p.execution_waves() == [["A"], ["B", "C"], ["D"]]
assert len(p.execution_waves()) == 3, "3 waves = the critical path length"
p2 = build(ParallelTaskScheduler, indep)
assert p2.execution_waves() == [["z", "y", "x"]], "fully independent tasks are one wave"

# Every wave's tasks must have all their dependencies in EARLIER waves
waves = p.execution_waves()
wave_of = {t: i for i, w in enumerate(waves) for t in w}
for tid, deps in DIAMOND:
    for d in deps:
        assert wave_of[d] < wave_of[tid], f"{d} must be in an earlier wave than {tid}"

# --- Eager cycle detection rejects the edge that closes the loop ---
e = EagerCycleScheduler()
for t in ("p", "q", "r"):
    e.add_task(t)
e.add_dependency("q", "p")                       # q depends on p
e.add_dependency("r", "q")                       # r depends on q
try:
    e.add_dependency("p", "r")                   # would close p -> q -> r -> p
except ValueError as exc:
    assert "cycle" in str(exc)
else:
    raise AssertionError("the edge that closes a cycle must be rejected at add time")
assert e.execute_all() == ["p", "q", "r"], "the graph is still valid after the rejection"

# --- Randomised DAGs: both implementations must produce valid orders ---
random.seed(19)
for _ in range(200):
    n = random.randint(1, 25)
    names = [f"t{i}" for i in range(n)]
    # Only depend on EARLIER tasks => acyclic by construction, and satisfies add_task's contract
    spec = [(names[i], random.sample(names[:i], k=random.randint(0, min(i, 3))))
            for i in range(n)]
    fast_order = build(TaskScheduler, spec).execute_all()
    naive_order = build(NaiveTaskScheduler, spec).execute_all()
    submission_order = build(SubmissionOrderScheduler, spec).execute_all()
    # All three are VALID topological orders; they need not be the SAME order, because
    # they use different tie-break policies among simultaneously-runnable tasks.
    for order in (fast_order, naive_order, submission_order):
        assert_valid_order(order, spec)
    # The naive scheduler always picks the earliest-submitted runnable task, which is
    # exactly what the heap-based scheduler does - so those two DO agree.
    assert naive_order == submission_order, "both use earliest-submitted tie-breaking"

    waves = build(ParallelTaskScheduler, spec).execution_waves()
    assert [t for w in waves for t in w] and sum(len(w) for w in waves) == n
    wave_of = {t: i for i, w in enumerate(waves) for t in w}
    for tid, deps in spec:
        for d in deps:
            assert wave_of[d] < wave_of[tid]

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Forward references (depending on a task not yet added).** Two contracts, both defensible. **Reject** (as here): the failure surfaces at the exact call that made the mistake, with a clear message. **Accept and defer**: record the edge in a pending map and resolve it when the task arrives — more convenient for config files where declaration order is arbitrary, but an unresolved reference then becomes indistinguishable from a cycle at execute time. If you defer, keep the pending edges separate so you can report *"never defined"* differently from *"circular"*.
- **Priorities or deadlines.** Swap the `deque` for a **heap** keyed by `(priority, submission_index)`. The topological constraint is untouched — a task still only enters the ready structure once its in-degree hits zero — you are just choosing differently *among* the currently-runnable set. Include the submission index in the key so equal priorities stay deterministic. Complexity becomes O(V log V + E).
- **Real parallel execution.** `execution_waves()` is the batched version, and it is simple but leaves capacity idle: a fast task in a wide wave sits waiting for the slowest sibling. The better scheduler is **continuous** — keep a worker pool pulling from the ready queue and decrement in-degrees as each task *completes*, so a new task can start the instant its last dependency finishes. The catch is that the decrement is now shared mutable state across threads and needs a lock (or an atomic counter); see [`4. Connection_Pool`](../4.%20Connection_Pool/4.%20Connection_Pool.ipynb) for the "keep the critical section tiny" discipline that applies.
- **A task that fails at runtime.** The graph tells you the blast radius exactly: everything **reachable** from the failed task must be skipped, and everything else can proceed. Compute it with one traversal of `dependents`. The policy choice — skip the subtree, retry with backoff (see [`18. Retry_Strategy`](../18.%20Retry_Strategy/README.md)), or halt everything — should be stated rather than assumed; build systems like Make and Bazel default to "skip the subtree, keep building the rest".
- **Kahn's vs. DFS-based topological sort.** DFS post-order reversed gives the same answer, and detects cycles via a three-colour visited marking (white/grey/black — a grey vertex reached again is a back edge). Kahn's is usually the better interview answer here: it is iterative (no recursion-depth limit), its cycle detection is a single length comparison, and — uniquely — it naturally exposes the *waves* of parallelism. DFS wins when you want the cycle **itself** printed out, since the grey path on the stack is the cycle.

## Empirical complexity check

Compare the **rescan** approach (Approach 1) with **Kahn's algorithm** (Approach 2) on a growing dependency graph where each task depends on up to three earlier ones.

| Growth when the task count doubles | What it means |
|---|---|
| ~4x | quadratic — every completion triggers a fresh scan of everything left |
| ~2x | linear in V + E — each task and each edge is touched exactly once |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random


def make_spec(n):
    rng = random.Random(23)
    names = [f"t{i}" for i in range(n)]
    # Each task depends on up to 3 earlier tasks => a realistic DAG, acyclic by construction.
    return ([(names[i], rng.sample(names[max(0, i - 20):i], k=min(i, rng.randint(0, 3))))
             for i in range(n)],)


def run_naive(spec):
    s = NaiveTaskScheduler()
    for tid, deps in spec:
        s.add_task(tid, deps)
    s.execute_all()                      # O(V^2): rescans after every completion


def run_kahn(spec):
    s = TaskScheduler()
    for tid, deps in spec:
        s.add_task(tid, deps)
    s.execute_all()                      # O(V + E): each task and edge touched once


benchmark(
    {"Approach 1 - rescan O(V^2)": run_naive,
     "Approach 2 - Kahn's algorithm O(V + E)": run_kahn},
    make_spec,
    sizes=[250, 500, 1000, 2000],
    repeats=2,
)

## Patterns learned

- **"Must happen before" is a directed edge.** Build dependencies, task ordering, course prerequisites, spreadsheet recalculation, package installation — all the same graph, all solved by topological sort.
- **Store the reverse edges too.** The forward direction expresses the constraint; the reverse direction is what you actually traverse when something completes. Building both at insert time is the whole difference between O(V²) and O(V + E).
- **Cache the answer to the question you keep asking.** "Is this runnable yet?" becomes an integer you decrement rather than a set-containment check you redo. That is the same move as an in-degree counter, a reference count, or a memoised result.
- **Kahn's gets cycle detection for free.** Anything in a cycle can never reach in-degree zero, so `executed < total` is the entire test — no second traversal, no colouring. Free correctness checks are worth recognising.
- **Fail at the point of the mistake.** Rejecting an unknown dependency in `add_task` gives an error that names the offending call. Deferring it to `execute_all` produces a message indistinguishable from a cycle.
- **Report *what* is broken, not just *that* it is.** `"Circular dependency among: [a, b, c]"` is actionable; `False` is not.
- **The queue's contents are the parallelism.** Everything at in-degree zero simultaneously is mutually independent. The number of waves is the critical path — the floor on completion time no matter how many workers you throw at it.
- **`deque`, not `list`, for a FIFO queue.** `list.pop(0)` shifts every element. One wrong container silently reintroduces a quadratic term.
- **Container choice decides determinism, not just speed.** A `set` for the reverse edges still yields a *correct* topological order — just a different one on every run. When output order is part of the contract, only insertion-ordered containers will do.